# పాఠం 18: క్రిప్టోగ్రాఫిక్ రసీదులతో AI ఏజెంట్లను సురక్షితం చేయడం

## హ్యాండ్-ఆన్ నోటుబుక్

ఈ నోటుబుక్ నాలుగు వ్యాయామాల ద్వారా నడిచిపోతుంది:

1. ఏజెంట్ టూల్ కాల్ కోసం మీ మొదటి రసీదును సంతకం చేయండి మరియు దాన్ని ధృవీకరించండి.
2. రసీదును చిమ్మండి మరియు ధృవీకరణ వైఫల్యాన్ని చూడండి.
3. మూడు-రసీదు గొలుసును నిర్మించి గొలుసు సమగ్రతను నిర్ధారించండి.
4. ప్రతి చర్య రసీదు విడుదల చేసే Microsoft Agent Framework టూల్ కాల్‌ను обుట్టండి.

అన్ని క్రిప్టోగ్రాఫిక్ ప్రీమిటివ్లు బాగా నిర్వహిత లైబ్రరీల నుండి దిగుమతి చేయబడ్డాయి (`pynacl` Ed25519 కోసం, `jcs` RFC 8785 కనానికల్ JSON కోసం, Python స్టాండర్డ్ లైబ్రరీ నుండి `hashlib` SHA-256 కోసం). రసీదు తత్త్వం స్వయంగా మీరు చదవగలిగే మరియు మార్చగల సమగ్ర పాథాన్ కోడ్.

క్రమం తప్పకుండా సెల్లులు నడపండి. ప్రతి విభాగం చిన్నది మరియు స్వీయసంపూర్ణం.


## సెటప్

రెండు డిపెండెన్సీలను ఇన్‌స్టాల్ చేయండి. రెండు పేరమిటివ్ లైసెన్సులు ఉన్నాయి (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## సహాయక ఉపకరణాలు

ఈ రెండు సహాయకులు బేస్64url ఎన్‌కోడ్ చేస్తాయి (ప్యాడింగ్ లేకుండా) మరియు అనుమతి పొందిన వస్తువుల SHA-256 హ్యాషింగ్ నిర్వహిస్తారు. అవి నోటుబుక్ లో నిలుపుదల తర్కంపై దృష్టి పెట్టడాన్ని కొనసాగిస్తాయి.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## సెక్షన్ 1: మీ మొదటి రసీదు పై సంతకం చేయండి

మన **Contoso Travel** ఏజెంట్ సిడ్నీ నుండి లాస్ ఏంజిల్స్ కు విమానాలను ఒక కస్టమర్ కోసం చూసుకున్నట్టు ఊహించండి. భవిష్యత్తులో ఒడిటర్ దాన్ని నమ్మకంతో కాకుండా ధృవీకరించగలిగేలా ఈ టూల్ కాల్ ను సంతకం చేయబడిన రసీదు గా నమోదు చేయదలిచాము.

### దశ 1.1: సంతకం కీని సృష్టించండి

ప్రతిష్టాత్మక వాతావరణంలో, ఏజెంట్ యొక్క సంతకం కీ హార్డ్వేర్ సెక్యూరిటీ మాడ్యూల్ (HSM), Azure Key Vault, లేదా ఇలాంటి రక్షిత నిల్వలో ఉంటాయి. ఈ పాఠం కోసం మనం మెమరీలో ఒక కొత్త కీని సృష్టిస్తాము.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### దశ 1.2: రసీదు పేలోడ్‌ను నిర్మించండి

పేలోడ్‌లో రసీదు నిర్ధారించవలసిన అన్నీ ఉంటాయి: ఎవరు చర్య తీసుకున్నారు, ఏ సాధనం, ఏ ఆర్గ్యుమెంట్లతో, ఏమి తిరిగి వచ్చింది, ఏ విధానం కింద, మరియు ఎప్పుడు. ఆర్గ్యుమెంట్లు మరియు ఫలితాన్ని inline లో చేర్చడం కాకుండా వాటిని హాష్ చేస్తాము అంటే రసీదు సున్నితమైన సమాచారం లీక్ కాకుండా ఉంటుంది.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: రసీదుపై సంతకం చేసి సమన్వయం చేయండి

మూడవస్ పాదాలు:

1. రెండు అమలు కలిగిన ఒకటే తర్కసంబంధమైన రసీదు తయారుచేసే రెండు వ్యవస్థలు బైట్-సరిగా సమానమైన బైట్లను ఉత్పత్తి చేయాలంటే JCS ఉపయోగించి పేతిన్ని Canonicalize చేయండి.
2. SHA-256 తో కెనానికల్ బైట్లకు హ్యాష్ చేయండి.
3. Ed25519 ప్రైవేట్ కీతో హ్యాష్‌పై సంతకం చేయండి.

అనంతరం సంతకం మూల పేతికి జత చేయబడుతుంది మరియు తుది రసీదు తయారవుతుంది.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### అడుగు 1.4: రసీదును ధృవీకరించండి

ధృవీకరణ ప్రక్రియను తిరగబడుతుంది. మేము సంతకాన్ని తొలగించి, ప్రామాణిక హాష్‌ను తిరిగి లెక్కించుకుంటాం, మరియు రసీదులోని పబ్లిక్ కీతో సంతకాన్ని పరిశీలిస్తాం.

ఈ ధృవీకరణను చేస్తున్న ఆడిటర్ మన నుండి అవసరమవుతోన్నది ఉన్నదంటే రసీదు మాత్రమే. ఏ సేవను కాల్ చేయాల్సిన అవసరం లేదు, ఏ కీ డైరెక్టరీని విచారించాల్సిన అవసరం లేదు, ఏ విశ్వాసం అవసరం లేదు.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

మీకు `Receipt is valid: True` కనిపిస్తుంది. ఏజెంట్ తన మొదటి క్రిప్టోగ్రాఫిక్‌గా సంతకం చేయబడిన ఆడిట్ రికార్డు ఉత్పత్తి చేసింది.


## విభాగం 2: రసీదు తుప్పుటి చేయండి

రసీదుల మొత్తం ఉద్దేశ్యం అవి తుప్పుటికి సాక్ష్యంగా ఉండటం. దీన్ని నిరూపిద్దాం.

మేము రసీదు లో ఖచ్చితంగా ఒక అక్షరాన్ని మార్చి ధృవీకరణ విఫలమవ్వటం వదిలి చూస్తాము.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ఏం జరిగిందంటే?

మేము `policy_id` మార్చినపుడు, కెనానికల్ బైట్లు మారాయి. ఆ బైట్ల SHA-256 హ్యాష్ మారింది. ఒiginaliginal hash పై ఉన్న సంతకం ఇప్పుడు కొత్త హ్యాష్‌తో సరిపోలదు. ధృవీకరణ సరిగ్గా `False` ను_Return_ చేస్తుంది.

రిసీట్ యొక్క ఏవైనా ఫీల్డ్ను మార్చి యధావిధిగా ధృవీకరించుకోవడం సాధ్యం కాదు, లెక్కలపాటి వ్యక్తికి ప్రైవేట్ కీ లేకపోతే. ప్రైవేట్ కీ కీ వాల్ట్‌లో ఉంటే మరియు పబ్లిక్ కీ ప్రచురించబడితే, దాడి దాచిపెట్టడం అసాధ్యం.

మీరు తెలివిగా ప్రయత్నించండి: పై సెల్‌లో `tool_name` లేదా `agent_id` లేదా `timestamp` మార్చి మళ్ళీ నడపండి. ప్రతి మార్పు చెల్లని రిసీట్‌ను ఉత్పత్తి చేస్తుంది.


## Section 3: రసీదులను చైన్లో కలపడం

ఒకే ఒక రసీదు ఒక చర్యను రక్షిస్తుంది. చాలా ఏజెంట్లు అనేక చర్యలు తీసుకుంటారు. మొత్తం వరుసను మోసపూరితంగా నిరోధించడానికి, మేము ప్రతి రసీదును ముందరి రసీదుతో కూడి లింక్ చేస్తాము, దీనిలో కొత్త రసీదులో గత రసీదుని హ్యాష్ ఉంటుంది.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ఎవరైనా ఒక రసీదును తీసివేస్తే లేదా క్రమాన్ని మార్చితే, చైన్ ఆ ఖచ్చితమైన చోటనే విరుగుతుంది. పైన ఉన్న ఏ రసీదును కూడా ధృవీకరించడం విఫలమవుతుంది ఎందుకంటే దాని `previous_receipt_hash` దాని ముందరి రసీదుని నిజమైన హ్యాష్ తో సరిపోదు.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ఇప్పుడు మధ్య మధ్య రసీదుతో చైన్‌ను ముట్టడి చేసి పునఃసరిపోయండి. ముట్టడి చేసిన రసీదు తన సంతకం తనిఖీను విఫలమవుతుంది, మరియు తదుపరి రసీదు దాని చైన్-లింక్ తనిఖీ విఫలమవుతుంది (దాని `previous_receipt_hash` మరింత మార్చబడిన మధ్య రసీదు యొక్క హాష్‌తో anymore సరిపోలకుండా ఉంటుంది).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 ఇంకా వాలిడేట్ అవుతుంది (దాన్ని ఎటువంటి మార్పు చేయలేదు మరియు దానికి ఆధారంగా ఆధిపత్యం ఉన్న ముందరి రసీదు లేదు). Receipt 1 తన సంతకం తనిఖీ విఫలమవుతుంది ఎందుకంటే మేము `tool_args_hash` ను మార్చాం. Receipt 2 తన చైన్-లింక్ తనిఖీ విఫలమవుతుంది ఎందుకంటే దాని `previous_receipt_hash` అసలు (ఇప్పుడప్పు మార్చబడిన) receipt 1 పై ఆధారపడి లెక్కించబడింది.

ఎంతైనా ఒక దాడిగాడు మార్చిన receipt 1 ను తిరిగి సంతకం చేస్తే (అది ప్రైవేట్ కీ లేకుండానే చేయలేడు), receipt 2 లో చైన్-లింక్ అసమతుల్యత ఇంకా మోసాన్ని బయటపెడుతుంది. మార్పును దాచడానికి, దాడిగాడు మార్పు జరిగిన పాయింట్ నుండి ప్రతి receipt ను తిరిగి సంతకం చేయాలి, ఇది ప్రైవేట్ కీ కలిగి ఉండాలి.


## Section 4: ఏజెంట్ టూల్ కాల్‌ను రశీదుతో సంతకం చేయడం ద్వారా లేపడం

యథార్థ సంస్థాపనలో, ప్రతి ఏజెంట్ రచయిత `make_receipt` కాల్ చేయాలని గుర్తుంచుకోవాలని లేదు. ప్రతి టూల్ ఆహ్వానం కోసం రశీదు సంతకం ఆటోమేటిక్‌గా ఉండాలని మీరు కోరుకుంటారు.

ఇప్పుడున్న సులభమైన నమూనా: ఏమైనా కాల్ చేయదగిన టూల్ ఫంక్షన్ తీసుకుని దానికి రశీదు విడుదల చేసే వెర్షన్‌ను తిరిగి ఇచ్చే ఓ రాపర్ క్లాస్. ఇది మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ (`agent_framework.azure`) సహా ఏ ఏజెంట్ ఫ్రేమ్‌వర్క్ కు సరిపోతుంది.

మీకు Azure AI Foundry ప్రాజెక్ట్ సెట్ చేయబడిపోలినట్లయితే, దిగువలో ఉన్న లోకల్ మాక్ ఈ నమూనాను ఇంకా చూపిస్తుంది.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్‌తో ఇంటిగ్రేట్ చేయడం

పైన ఉన్న `ReceiptedTool` ర్యాపర్ ఫ్రేమ్‌వర్క్-అగ్నోస్టిక్. దీన్ని Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్‌తో నిర్మించబడిన ఏజెంట్‌లో ఉపయోగించడానికి, ర్యాప్ చేసిన ఫังก్షన్‌ను టూల్‌గా రిజిస్టర్ చేయండి. ఒక స్కెచ్ (మీరు మాక్‌ను నిజమైన Azure AI Foundry టూల్ రిజిస్ట్రేషన్‌తో స్థానపూర్వకంగా మార్చాలి):

```python
# సమీకరణ ఆకారాన్ని చూపించే అర్థరూపి కోడ్.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="మీరు కాంటోసో ట్రావెల్ ఏజెన్ట్ ...",
#     tools=[receipted_lookup],   # ఆ వర్తించిన టూల్, కచ్చితమైన ఫంక్షన్ కాదు
# )
# response = agent.run("జూన్ లో సిడ్నీ నుండి లాస్ ఏంజిల్స్ కు ఫ్లైట్లు కనుగొనండి.")
#
# # రన్ తర్వాత, ఏజెంట్ చేసిన ప్రతి టూల్ కాల్ కు సంతకమున్న రసీదు ఉంటుంది:
# audit_chain = receipted_lookup.receipts
```

ఏజెంట్ ఫ్రేమ్‌వర్క్‌కు రసీదులు గురించి ఏమీ తెలుసుకోవాల్సిన అవసరం లేదు. రసీదు సంతకం టూల్ చుట్టూ ర్యాప్ చేయబడింది, ఫ్రేమ్‌వర్క్‌లో పెడలేదు. మీరు ఏజెంట్‌ను మళ్లీ రాయకుండా ప్రొవెనెన్స్‌ను ఇక్కడున్న ఏజెంట్ కోడ్‌కు ఎలా జోడించాలో ఇదే.


## సారాంశం మరియు సవాలు

మీకు:

- Ed25519 కీ జంటను సృష్టించారు.
- ఏజెంట్ టూల్ కాల్ కోసం రసీదును నిర్మించి సంతకం చేశారు.
- రసీదును ఆఫ్‌లైన్‌లో పబ్లిక్ కీ ద్వారా ధృవీకరించారు.
- రసీదును మోసపూరితంగా మార్చి ధృవీకరణ విఫలమైందని చూశారు.
- మూడు రసీదుల హ్యాష్-చైన్స్ సీక్వెన్స్ నిర్మించారు.
- చైన్ మధ్య భాగంలో మోసం చేసి సంతకం విఫలం మరియు చైన్-లింక్ విఫలం రెండూ కలిగినట్లు చూశారు.
- టూల్ ఫంక్షన్ ను స్వయంచాలక రసీదు సంతకంతో.Wrap చేశారు.

**సవాలుకి విస్తరణ.** రసీదు పటానికి `request_id` ఫీల్డ్ (విభజిత ట్రేసింగ్ కు UUID)ని జోడించండి. `make_receipt` ను ఈ ఫీల్డ్ తో నవీకరించడం మరియు రసీదులు ఎండ్-టు-ఎండ్ ధృవీకరించబడతాయా అని నిర్ధారించండి. ఆపై సంతకం తర్వాత ఫీల్డ్ ను మార్చి ధృవీకరణ విఫలమవుతుందో చెక్ చేయండి. ఇది ప్రతి బైట్ ఎలా సంతకానికి సహకరించారో బాగా అవగాహన పొందడానికి భలే.

**ముఖ్యమైన సరిహద్దు.** రసీదులు మూడు విషయాలను మాత్రమే నిరూపిస్తాయి: అప్పగింపు (ఈ కీ సంతకం చేసినది), సారూప్యత (సంతకం తరువాత విషయము మారలేదు), మరియు క్రమపద్ధతులు (ఈ రసీదు ఆ రసీదు తరువాత వచ్చింది). అవి ఏజెంట్ చర్య సరైనదైందని, `policy_id` లో ఉన్న విధానం నిజంగా అమలు అయ్యిందని, లేదా ఏజెంట్ ప్రతి నియమాన్ని పాటించిందని నిరూపించవు. రసీదులు ఒక పునాది మాత్రమే. పాలన మీరు నిర్మించే వ్యవస్థ.

ఆ సరిహద్దు గుర్తుంచుకొని పాఠం README ను మళ్ళీ చదవండి. రసీదులు ఉన్నామని అంటే "మనం పాలనలో ఉన్నాం" అని భావించడం సాధారణ లోపం. రసీదులు ఏజెంట్ ప్రవర్తనను ఆడిట్ చేయదగినది చేస్తాయి. అవి దానిని సరైనదిగా మార్చవు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
